In [1]:
# Cell 1 — Install and Import Libraries
#
# Colab comes with TensorFlow, NumPy, Matplotlib pre-installed.
# We only need to install tf-keras-vis which is not pre-installed.
# The ! prefix means "run this as a terminal command, not Python code"

!pip install tf-keras-vis -q  # -q means quiet mode — suppresses long install logs

# --- Standard library ---
import os           # File and folder navigation
import random       # For shuffling and sampling
import numpy as np  # Array and matrix operations

# --- Visualization ---
import matplotlib.pyplot as plt  # Plotting training curves and results
import seaborn as sns            # Prettier confusion matrices

# --- TensorFlow / Keras ---
# TensorFlow is the engine. Keras is the user-friendly layer on top of it.
import tensorflow as tf
from tensorflow import keras

# These are the three pretrained backbones we'll use for our ablation study
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50

# Layers we'll use to build our custom classification head
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,  # Converts feature maps to a flat vector
    Dense,                   # Fully connected layer
    Dropout,                 # Randomly drops neurons during training to prevent overfitting
    BatchNormalization       # Normalizes layer inputs — speeds up training
)

# Model building utilities
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam  # Our optimization algorithm
from tensorflow.keras.callbacks import (
    EarlyStopping,    # Stops training when validation stops improving
    ModelCheckpoint,  # Saves best model automatically during training
    ReduceLROnPlateau # Reduces learning rate when training plateaus
)

# Image data loading and augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix

# --- Verify GPU is active ---
# This should print something like: [PhysicalDevice(name='/physical_device:GPU:0', ...)]
# If it prints an empty list [], go to Runtime → Change runtime type → GPU
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)
print("\n✅ All libraries loaded successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 3.4 MB/s eta 0:00:00
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.19.0

✅ All libraries loaded successfully


In [3]:
# Cell 2 — Mount Google Drive and Set Paths
#
# Mounting makes your Google Drive accessible as a folder at /content/drive/
# When you run this cell, Google will ask you to authorize access.
# Click the link, choose your account, copy the code, paste it back here.

from google.colab import drive
drive.mount('/content/drive')

# --- Set the path to your uploaded processed_data folder ---
# Adjust this if you named your Drive folder differently
# My Drive is always at /content/drive/MyDrive/

DATA_DIR = '/content/drive/MyDrive/FloraGuard_Data/processed_data'

# Verify the path is correct by listing what's inside
print("Contents of processed_data/:")
for item in sorted(os.listdir(DATA_DIR)):
    print(f"  📁 {item}")

# --- Set path for saving trained models ---
# We'll save directly to Drive so models survive after Colab session ends
# Colab's local storage (/content/) gets wiped when session closes
MODEL_DIR = '/content/drive/MyDrive/FloraGuard_Data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Global training configuration ---
# Defined once here, used everywhere below
# Changing these values here automatically updates the entire notebook

IMAGE_SIZE   = (224, 224)  # Must match what preprocess.py output
BATCH_SIZE   = 32          # How many images processed together per step
                           # 32 is standard — fits in GPU memory comfortably
EPOCHS       = 20          # Maximum training rounds
                           # EarlyStopping will likely stop before this
LEARNING_RATE = 0.0001     # How big each weight update step is
                           # 0.0001 is conservative — safe for transfer learning
NUM_CLASSES  = 8           # 7 PlantVillage classes + 1 Rose_Disease from garden
                           # Wait — we'll verify the actual count in Cell 3

print(f"\n✅ Drive mounted successfully")
print(f"   Data directory : {DATA_DIR}")
print(f"   Models directory: {MODEL_DIR}")

Mounted at /content/drive
Contents of processed_data/:
  📁 My_Garden
  📁 PlantVillage

✅ Drive mounted successfully
   Data directory : /content/drive/MyDrive/FloraGuard_Data/processed_data
   Models directory: /content/drive/MyDrive/FloraGuard_Data/models


In [4]:
# Cell 3 — Merge and Flatten Dataset Structure
#
# Creates a flat class-folder structure that Keras can read directly.
# This runs entirely in Colab's local storage (/content/floragard_flat/)
# so it's fast — no Drive read/write bottleneck during training.
#
# Final structure created:
#   /content/floragard_flat/
#       Hibiscus_Healthy/          (15 images)
#       Rose_Disease/              (15 images)
#       Parijat_Healthy/           (15 images)
#       Pepper__bell___Bacterial_spot/ (997 images)
#       Pepper__bell___healthy/    (1478 images)
#       Potato___Early_blight/     (1000 images)
#       Potato___Late_blight/      (1000 images)
#       Potato___healthy/          (152 images)
#       Tomato_Bacterial_spot/     (2127 images)
#       Tomato_healthy/            (1591 images)

import shutil  # shutil = shell utilities — handles file copy/move operations

FLAT_DIR = '/content/floragard_flat'

# Remove flat dir if it already exists (in case you're rerunning this cell)
if os.path.exists(FLAT_DIR):
    shutil.rmtree(FLAT_DIR)
    print("♻️  Removed existing flat directory")

os.makedirs(FLAT_DIR)
print(f"📁 Created: {FLAT_DIR}")

# --- Walk through ALL subfolders in processed_data ---
# For each folder that contains images, copy those images to FLAT_DIR/class_name/
# The class name is simply the innermost folder name
# e.g. processed_data/My_Garden/Rose_Disease/ → class name = "Rose_Disease"

total_copied = 0
class_counts = {}  # Track how many images per class

for root, dirs, files in os.walk(DATA_DIR):

    # Filter image files only
    image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if not image_files:
        continue  # Skip folders with no images

    # The class name is the name of the innermost folder
    # os.path.basename returns the last component of a path
    # e.g. os.path.basename('/content/drive/.../Rose_Disease') = 'Rose_Disease'
    class_name = os.path.basename(root)

    # Create this class's folder in the flat directory
    class_flat_dir = os.path.join(FLAT_DIR, class_name)
    os.makedirs(class_flat_dir, exist_ok=True)

    # Copy every image into the flat class folder
    for filename in image_files:
        src = os.path.join(root, filename)
        dst = os.path.join(class_flat_dir, filename)
        shutil.copy2(src, dst)  # copy2 preserves file metadata
        total_copied += 1

    class_counts[class_name] = len(image_files)
    print(f"  ✓ {class_name:<45} {len(image_files):>5} images")

print(f"\n{'='*55}")
print(f"  Total images copied to flat directory: {total_copied}")
print(f"  Number of classes: {len(class_counts)}")
print(f"{'='*55}")

# Update NUM_CLASSES to actual count
NUM_CLASSES = len(class_counts)
print(f"\n✅ NUM_CLASSES updated to: {NUM_CLASSES}")

📁 Created: /content/floragard_flat
  ✓ Tomato_healthy                                 1592 images
  ✓ Tomato_Bacterial_spot                          2127 images
  ✓ Potato___Early_blight                          1000 images
  ✓ Potato___healthy                                152 images
  ✓ Potato___Late_blight                           1000 images
  ✓ Pepper__bell___healthy                         1478 images
  ✓ Pepper__bell___Bacterial_spot                   997 images
  ✓ Rose_Disease                                     15 images
  ✓ Hibiscus_Healthy                                 15 images
  ✓ Parijat_Healthy                                  15 images

  Total images copied to flat directory: 8391
  Number of classes: 10

✅ NUM_CLASSES updated to: 10


In [5]:
# Cell 4 — Load Data and Create Train/Validation/Test Splits
#
# ImageDataGenerator does two things:
#   1. Loads images from folders in batches (memory efficient — not all at once)
#   2. Applies data augmentation on-the-fly during training
#
# Why augmentation matters for our custom dataset:
#   We only have 15 images per garden class. Without augmentation,
#   the model sees the same 15 images every epoch — pure overfitting.
#   Augmentation artificially creates variations of each image:
#   flipped, rotated, brightness-shifted — each epoch the model sees
#   slightly different versions, forcing it to generalize.

# --- Training data generator WITH augmentation ---
# These transforms are applied RANDOMLY to each image each time it's loaded
# The original image on disk is never modified — transforms happen in memory

train_datagen = ImageDataGenerator(
    rescale=1./255,          # Converts pixel values from [0,255] to [0,1]
                             # Neural networks train much better with small input values
    validation_split=0.2,    # Reserve 20% of data for validation
                             # The remaining 80% is used for training

    # --- Augmentation transforms (training only) ---
    horizontal_flip=True,    # Randomly mirror image left-right
                             # A diseased leaf looks the same flipped — valid transform
    vertical_flip=False,     # Don't flip upside down — leaves have a natural orientation
    rotation_range=20,       # Randomly rotate ±20 degrees
    zoom_range=0.15,         # Randomly zoom in/out by up to 15%
    width_shift_range=0.1,   # Randomly shift image horizontally by up to 10%
    height_shift_range=0.1,  # Randomly shift image vertically by up to 10%
    brightness_range=[0.8, 1.2],  # Randomly adjust brightness ±20%
                                   # Simulates different lighting conditions
    fill_mode='nearest'      # When transforms create empty pixels at edges,
                             # fill them by copying nearest pixel value
)

# --- Validation data generator WITHOUT augmentation ---
# We never augment validation or test data — we want consistent, unmodified
# images for evaluation so results are comparable across runs
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2     # Must match train_datagen's validation_split
)

# --- Load training data ---
# flow_from_directory reads images from our flat folder structure
# Each subfolder name automatically becomes a class label
# subset='training' loads the 80% training portion

train_generator = train_datagen.flow_from_directory(
    FLAT_DIR,
    target_size=IMAGE_SIZE,   # Resize to 224×224 (already done but this ensures consistency)
    batch_size=BATCH_SIZE,    # 32 images per batch
    class_mode='categorical', # One-hot encoded labels (e.g. [0,1,0,0,0,0,0,0] for class 2)
    subset='training',        # Use the 80% training split
    shuffle=True,             # Shuffle order every epoch — prevents order-based patterns
    seed=42                   # Fixed seed = reproducible shuffling across runs
)

# --- Load validation data ---
val_generator = val_datagen.flow_from_directory(
    FLAT_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',      # Use the 20% validation split
    shuffle=False,            # Don't shuffle validation — consistent evaluation
    seed=42
)

# --- Print dataset summary ---
print(f"\n📊 Dataset Summary:")
print(f"   Total training images  : {train_generator.samples}")
print(f"   Total validation images: {val_generator.samples}")
print(f"   Number of classes      : {len(train_generator.class_indices)}")
print(f"\n📋 Class → Label mapping:")
for class_name, index in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    print(f"   {index}: {class_name}")

Found 6715 images belonging to 10 classes.
Found 1676 images belonging to 10 classes.

📊 Dataset Summary:
   Total training images  : 6715
   Total validation images: 1676
   Number of classes      : 10

📋 Class → Label mapping:
   0: Hibiscus_Healthy
   1: Parijat_Healthy
   2: Pepper__bell___Bacterial_spot
   3: Pepper__bell___healthy
   4: Potato___Early_blight
   5: Potato___Late_blight
   6: Potato___healthy
   7: Rose_Disease
   8: Tomato_Bacterial_spot
   9: Tomato_healthy


In [11]:
# Cell 5 (Updated) — Model Builder With Backbone-Specific Preprocessing
#
# Each backbone has its own preprocess_input function that was used
# during its original ImageNet training. We must use the SAME
# preprocessing at inference time — that's what these imports provide.
#
# What each preprocess_input does:
#   MobileNetV2   → scales pixels to [-1, 1]
#   EfficientNetB0 → scales pixels to [0, 1] with its own mean subtraction
#   ResNet50      → subtracts ImageNet mean RGB values per channel
#
# By adding it as the FIRST layer inside the model, preprocessing
# is baked into the model itself. The model becomes self-contained —
# it accepts raw [0,255] pixel images and handles normalization internally.
# This also makes deployment cleaner — the Streamlit app just passes
# raw images without any manual preprocessing step.

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.layers import Lambda

def build_model(backbone_name):
    """
    Builds a transfer learning model with backbone-specific preprocessing
    baked in as the first layer.

    Parameters:
        backbone_name (str): One of 'MobileNetV2', 'EfficientNetB0', 'ResNet50'

    Returns:
        model: Compiled Keras model ready for training
    """

    # Map backbone names to their classes and preprocessing functions
    backbone_config = {
        'MobileNetV2':    (MobileNetV2,    mobilenet_preprocess),
        'EfficientNetB0': (EfficientNetB0, efficientnet_preprocess),
        'ResNet50':       (ResNet50,        resnet_preprocess)
    }

    if backbone_name not in backbone_config:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    BackboneClass, preprocess_fn = backbone_config[backbone_name]

    # --- Step 1: Define input layer ---
    # We explicitly define the input so we can insert preprocessing before the backbone
    # Shape: (224, 224, 3) — height, width, RGB channels
    inputs = keras.Input(shape=(224, 224, 3))

    # --- Step 2: Apply backbone-specific preprocessing ---
    # Lambda wraps any Python function as a Keras layer
    # This means preprocessing becomes part of the model graph —
    # it runs automatically every time the model processes an image
    x = Lambda(
        lambda img: preprocess_fn(img),
        name='backbone_preprocessing'
    )(inputs)

    # --- Step 3: Load pretrained backbone ---
    backbone = BackboneClass(
        include_top=False,
        weights='imagenet',
        input_tensor=x    # Connect backbone directly to preprocessed input
    )

    # --- Step 4: Freeze backbone layers ---
    backbone.trainable = False

    print(f"\n🏗  Building {backbone_name}...")
    print(f"   Backbone layers  : {len(backbone.layers)}")
    print(f"   Preprocessing    : {preprocess_fn.__module__} ✓")
    print(f"   Backbone frozen  : ✓")

    # --- Step 5: Classification head ---
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    # --- Step 6: Build and compile ---
    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    total_params     = model.count_params()
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

    print(f"   Total parameters    : {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    print(f"   Frozen parameters   : {total_params - trainable_params:,}")
    print(f"   ✅ {backbone_name} ready for training\n")

    return model

In [7]:
# Cell 6 — Define Training Callbacks
#
# Callbacks are functions that Keras calls automatically at specific points
# during training (end of each epoch, end of each batch, etc.)
# We use three:

def get_callbacks(model_name):
    """
    Returns a list of callbacks for training a given model.

    Parameters:
        model_name (str): Used for naming the saved checkpoint file

    Returns:
        list of Keras callback objects
    """

    # --- Callback 1: EarlyStopping ---
    # Monitors validation accuracy after every epoch.
    # If it doesn't improve for 5 consecutive epochs → stops training.
    # This prevents wasting GPU time and prevents overfitting.
    # restore_best_weights=True means after stopping, weights revert
    # to the epoch where validation accuracy was highest — not the last epoch.
    early_stop = EarlyStopping(
        monitor='val_accuracy',   # Watch validation accuracy
        patience=5,               # Stop after 5 epochs of no improvement
        restore_best_weights=True,
        verbose=1                 # Print a message when it triggers
    )

    # --- Callback 2: ModelCheckpoint ---
    # Saves the model to Drive automatically whenever validation accuracy improves.
    # Even if Colab crashes mid-training, you won't lose your best weights.
    # save_best_only=True means only saves if this epoch is better than all previous.
    checkpoint_path = os.path.join(MODEL_DIR, f'{model_name}_best.keras')
    checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    # --- Callback 3: ReduceLROnPlateau ---
    # If validation loss doesn't improve for 3 epochs, reduce learning rate by 50%.
    # Think of it like this: if you're lost hiking and keep hitting dead ends,
    # you take smaller steps to navigate more carefully.
    # min_lr=1e-7 ensures the learning rate never drops to zero.
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # Multiply learning rate by 0.5 when triggered
        patience=3,
        min_lr=1e-7,
        verbose=1
    )

    return [early_stop, checkpoint, reduce_lr]

print("✅ Callback factory defined")
print("   EarlyStopping    — stops when val_accuracy stagnates for 5 epochs")
print("   ModelCheckpoint  — saves best model to Drive automatically")
print("   ReduceLROnPlateau — halves learning rate when val_loss plateaus for 3 epochs")

✅ Callback factory defined
   EarlyStopping    — stops when val_accuracy stagnates for 5 epochs
   ModelCheckpoint  — saves best model to Drive automatically
   ReduceLROnPlateau — halves learning rate when val_loss plateaus for 3 epochs


In [12]:
# Cell 7 — Class Weights + Train MobileNetV2
#
# This cell does three things:
#   1. Calculates class weights to handle imbalance
#   2. Builds the MobileNetV2 model using our builder function
#   3. Trains it and stores the training history

# ===========================================================
# PART A: Calculate Class Weights
# ===========================================================

# sklearn's compute_class_weight does the formula automatically
from sklearn.utils.class_weight import compute_class_weight

# We need the class label for every image in the training set
# train_generator.classes is a NumPy array like [0, 0, 1, 2, 2, 3, ...]
# where each number is the class index of one training image
training_labels = train_generator.classes

# np.unique returns the sorted unique class indices: [0, 1, 2, ..., 9]
class_indices = np.unique(training_labels)

# compute_class_weight calculates the weight for each class
# 'balanced' mode uses our formula: total / (n_classes × class_count)
weights = compute_class_weight(
    class_weight='balanced',
    classes=class_indices,
    y=training_labels
)

# Convert to a dictionary {class_index: weight}
# Keras expects this exact format for class_weight parameter in model.fit()
class_weight_dict = dict(zip(class_indices, weights))

# Print weights so we can verify the logic
print("📊 Class Weights:")
class_names = list(train_generator.class_indices.keys())
for idx, weight in class_weight_dict.items():
    bar = "█" * min(int(weight * 2), 40)  # Visual bar capped at 40 chars
    print(f"   {idx}: {class_names[idx]:<40} weight={weight:.3f}  {bar}")

# ===========================================================
# PART B: Build MobileNetV2
# ===========================================================
# Calling build_model() for the first time — this is when the
# print statements inside it actually run

mobilenet_model = build_model('MobileNetV2')

# ===========================================================
# PART C: Train MobileNetV2
# ===========================================================

print("🚀 Starting MobileNetV2 training...")
print(f"   Epochs (max)     : {EPOCHS}")
print(f"   Batch size       : {BATCH_SIZE}")
print(f"   Learning rate    : {LEARNING_RATE}")
print(f"   Early stopping   : patience=5\n")

# model.fit() is the actual training call
# Everything we've built leads to this one line
#
# Parameters explained:
#   train_generator     — our augmented training data, loaded in batches
#   epochs              — maximum number of full passes through training data
#   validation_data     — checked after every epoch, never trained on
#   class_weight        — our imbalance fix dictionary
#   callbacks           — EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
#   verbose=1           — print one progress bar per epoch (not per batch)

mobilenet_history = mobilenet_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('mobilenet'),
    verbose=1
)

# Training complete — print final results
final_train_acc = mobilenet_history.history['accuracy'][-1]
final_val_acc   = mobilenet_history.history['val_accuracy'][-1]
best_val_acc    = max(mobilenet_history.history['val_accuracy'])
epochs_run      = len(mobilenet_history.history['accuracy'])

print(f"\n{'='*55}")
print(f"  MobileNetV2 Training Complete")
print(f"{'='*55}")
print(f"  Epochs run          : {epochs_run}/{EPOCHS}")
print(f"  Final train accuracy: {final_train_acc:.4f} ({final_train_acc*100:.1f}%)")
print(f"  Final val accuracy  : {final_val_acc:.4f} ({final_val_acc*100:.1f}%)")
print(f"  Best val accuracy   : {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
print(f"  Model saved to      : {MODEL_DIR}/mobilenet_best.keras")
print(f"{'='*55}")

📊 Class Weights:
   0: Hibiscus_Healthy                         weight=55.958  ████████████████████████████████████████
   1: Parijat_Healthy                          weight=55.958  ████████████████████████████████████████
   2: Pepper__bell___Bacterial_spot            weight=0.841  █
   3: Pepper__bell___healthy                   weight=0.568  █
   4: Potato___Early_blight                    weight=0.839  █
   5: Potato___Late_blight                     weight=0.839  █
   6: Potato___healthy                         weight=5.504  ███████████
   7: Rose_Disease                             weight=55.958  ████████████████████████████████████████
   8: Tomato_Bacterial_spot                    weight=0.395  
   9: Tomato_healthy                           weight=0.527  █


/tmp/ipython-input-3525756825.py:62: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = BackboneClass(



🏗  Building MobileNetV2...
   Backbone layers  : 155
   Preprocessing    : keras.src.applications.mobilenet_v2 ✓
   Backbone frozen  : ✓
   Total parameters    : 2,593,610
   Trainable parameters: 333,066
   Frozen parameters   : 2,260,544
   ✅ MobileNetV2 ready for training

🚀 Starting MobileNetV2 training...
   Epochs (max)     : 20
   Batch size       : 32
   Learning rate    : 0.0001
   Early stopping   : patience=5



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 466ms/step - accuracy: 0.3475 - loss: 2.8371
Epoch 1: val_accuracy improved from -inf to 0.81742, saving model to /content/drive/MyDrive/FloraGuard_Data/models/mobilenet_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 121s 522ms/step - accuracy: 0.3482 - loss: 2.8332 - val_accuracy: 0.8174 - val_loss: 0.7824 - learning_rate: 1.0000e-04
Epoch 2/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.7453 - loss: 0.8740
Epoch 2: val_accuracy improved from 0.81742 to 0.88186, saving model to /content/drive/MyDrive/FloraGuard_Data/models/mobilenet_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 97s 461ms/step - accuracy: 0.7454 - loss: 0.8741 - val_accuracy: 0.8819 - val_loss: 0.4335 - learning_rate: 1.0000e-04
Epoch 3/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.8305 - loss: 0.5749
Epoch 3: val_accuracy improved from 0.88186 to 0.91945, saving model to /content/drive/MyDrive/FloraGuard_Data/models/mobilenet_best.keras
210/210 ━━━━━━━━━━━━━━━━━

In [13]:
# Cell 8 — Train EfficientNetB0
#
# EfficientNetB0 uses a technique called "compound scaling" —
# it scales network depth, width, and resolution simultaneously
# rather than just making the network deeper (like ResNet) or
# just wider. This makes it more parameter-efficient than ResNet50
# while typically being more accurate than MobileNetV2.
#
# One important difference from MobileNetV2:
# EfficientNet was designed expecting pixel values in [0, 255] range
# internally — it has its own normalization built in.
# However since we already rescale to [0,1] in our ImageDataGenerator,
# it still works correctly. No changes needed to our data pipeline.

# Build EfficientNetB0 using our same builder function
efficientnet_model = build_model('EfficientNetB0')

print("🚀 Starting EfficientNetB0 training...")
print(f"   Epochs (max)     : {EPOCHS}")
print(f"   Batch size       : {BATCH_SIZE}")
print(f"   Learning rate    : {LEARNING_RATE}")
print(f"   Early stopping   : patience=5\n")

efficientnet_history = efficientnet_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,   # Same class weights as MobileNetV2
    callbacks=get_callbacks('efficientnet'),  # Fresh callbacks for this model
    verbose=1
)

# --- Print final results ---
final_train_acc = efficientnet_history.history['accuracy'][-1]
final_val_acc   = efficientnet_history.history['val_accuracy'][-1]
best_val_acc    = max(efficientnet_history.history['val_accuracy'])
epochs_run      = len(efficientnet_history.history['accuracy'])

print(f"\n{'='*55}")
print(f"  EfficientNetB0 Training Complete")
print(f"{'='*55}")
print(f"  Epochs run          : {epochs_run}/{EPOCHS}")
print(f"  Final train accuracy: {final_train_acc:.4f} ({final_train_acc*100:.1f}%)")
print(f"  Final val accuracy  : {final_val_acc:.4f} ({final_val_acc*100:.1f}%)")
print(f"  Best val accuracy   : {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
print(f"  Model saved to      : {MODEL_DIR}/efficientnet_best.keras")
print(f"{'='*55}")

# --- Running comparison so far ---
print(f"\n📊 Ablation Study — Results So Far:")
print(f"   MobileNetV2    : 97.0% val accuracy")
print(f"   EfficientNetB0 : {best_val_acc*100:.1f}% val accuracy")
print(f"   ResNet50       : (pending)")


🏗  Building EfficientNetB0...
   Backbone layers  : 239
   Preprocessing    : keras.src.applications.efficientnet ✓
   Backbone frozen  : ✓
   Total parameters    : 4,385,197
   Trainable parameters: 333,066
   Frozen parameters   : 4,052,131
   ✅ EfficientNetB0 ready for training

🚀 Starting EfficientNetB0 training...
   Epochs (max)     : 20
   Batch size       : 32
   Learning rate    : 0.0001
   Early stopping   : patience=5

Epoch 1/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 0.3265 - loss: 2.4753
Epoch 1: val_accuracy improved from -inf to 0.85143, saving model to /content/drive/MyDrive/FloraGuard_Data/models/efficientnet_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 141s 568ms/step - accuracy: 0.3274 - loss: 2.4731 - val_accuracy: 0.8514 - val_loss: 0.9781 - learning_rate: 1.0000e-04
Epoch 2/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.7678 - loss: 0.8813
Epoch 2: val_accuracy improved from 0.85143 to 0.92721, saving model to /content/drive/MyDrive/FloraG

In [10]:
# Cell 4 (Updated) — Data Generators WITHOUT rescale
#
# We remove rescale=1./255 entirely.
# Each model will handle its own normalization internally
# via its backbone-specific preprocess_input function.
# This is the correct, professional way to use pretrained backbones.

train_datagen = ImageDataGenerator(
    # NO rescale here anymore
    validation_split=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    # NO rescale here either
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    FLAT_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    FLAT_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f"✅ Data generators recreated (no rescaling — backbones handle their own preprocessing)")
print(f"   Training images  : {train_generator.samples}")
print(f"   Validation images: {val_generator.samples}")

Found 6715 images belonging to 10 classes.
Found 1676 images belonging to 10 classes.
✅ Data generators recreated (no rescaling — backbones handle their own preprocessing)
   Training images  : 6715
   Validation images: 1676


In [14]:
# Cell 9 — Train ResNet50
#
# ResNet50 introduced "residual connections" (skip connections) in 2015 —
# a revolutionary idea that let networks be much deeper without the
# vanishing gradient problem that plagued earlier deep networks.
#
# "Vanishing gradient" means: in very deep networks, the error signal
# that flows backward during training gets multiplied by small numbers
# at each layer until it essentially becomes zero by the time it
# reaches early layers. Those early layers then stop learning.
#
# ResNet's fix: add a "shortcut" connection that skips 2-3 layers,
# carrying the gradient directly backward without it passing through
# those intermediate multiplications. Gradients stay healthy deep
# into the network.
#
# ResNet50 has 50 layers and ~25M parameters — significantly heavier
# than MobileNetV2 (2.3M) and EfficientNetB0 (4M).
# More parameters doesn't always mean better accuracy though —
# that's exactly what this ablation study is proving.

resnet_model = build_model('ResNet50')

print("🚀 Starting ResNet50 training...")
print(f"   Epochs (max)     : {EPOCHS}")
print(f"   Batch size       : {BATCH_SIZE}")
print(f"   Learning rate    : {LEARNING_RATE}")
print(f"   Early stopping   : patience=5\n")

resnet_history = resnet_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('resnet'),
    verbose=1
)

# --- Final results ---
final_train_acc = resnet_history.history['accuracy'][-1]
final_val_acc   = resnet_history.history['val_accuracy'][-1]
best_val_acc    = max(resnet_history.history['val_accuracy'])
epochs_run      = len(resnet_history.history['accuracy'])

print(f"\n{'='*55}")
print(f"  ResNet50 Training Complete")
print(f"{'='*55}")
print(f"  Epochs run          : {epochs_run}/{EPOCHS}")
print(f"  Final train accuracy: {final_train_acc:.4f} ({final_train_acc*100:.1f}%)")
print(f"  Final val accuracy  : {final_val_acc:.4f} ({final_val_acc*100:.1f}%)")
print(f"  Best val accuracy   : {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
print(f"  Model saved to      : {MODEL_DIR}/resnet_best.keras")
print(f"{'='*55}")

# --- Complete ablation study summary ---
mobilenet_best    = max(mobilenet_history.history['val_accuracy'])
efficientnet_best = max(efficientnet_history.history['val_accuracy'])
resnet_best       = best_val_acc

print(f"\n{'='*55}")
print(f"  🏆 ABLATION STUDY — COMPLETE RESULTS")
print(f"{'='*55}")
print(f"  {'Model':<20} {'Val Accuracy':>14} {'Rank':>6}")
print(f"  {'-'*44}")

results = [
    ('MobileNetV2',    mobilenet_best),
    ('EfficientNetB0', efficientnet_best),
    ('ResNet50',       resnet_best)
]

# Sort by accuracy descending to assign ranks
ranked = sorted(results, key=lambda x: x[1], reverse=True)
rank_map = {name: i+1 for i, (name, _) in enumerate(ranked)}

for name, acc in results:
    rank  = rank_map[name]
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
    print(f"  {name:<20} {acc*100:>13.1f}%  {medal}")

print(f"{'='*55}")
winner = ranked[0][0]
print(f"\n  ✅ Best model: {winner}")
print(f"  This model will be used for Grad-CAM and Streamlit deployment.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step

🏗  Building ResNet50...
   Backbone layers  : 176
   Preprocessing    : keras.src.applications.resnet ✓
   Backbone frozen  : ✓
   Total parameters    : 24,123,018
   Trainable parameters: 531,210
   Frozen parameters   : 23,591,808
   ✅ ResNet50 ready for training

🚀 Starting ResNet50 training...
   Epochs (max)     : 20
   Batch size       : 32
   Learning rate    : 0.0001
   Early stopping   : patience=5

Epoch 1/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step - accuracy: 0.5205 - loss: 2.2304
Epoch 1: val_accuracy improved from -inf to 0.90394, saving model to /content/drive/MyDrive/FloraGuard_Data/models/resnet_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 131s 560ms/step - accuracy: 0.5212 - loss: 2.2277 - val_accuracy: 0.9039 - val_loss: 0.4763 - learning_rate: 1.0000e-04
Epoch 2/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step - accuracy: 0.8386 - loss: 0.7818
Epoch 2: val_accuracy improved from 0.90394 to 0.94570, saving model to /conte